## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and connects the data from Google Drive.

**Before you run it**, make sure you've opened the shared camp Drive folder and
clicked **"Add shortcut to Drive"** (put the shortcut in *My Drive*) — that's how
the notebook finds the data file. Then run the cell below and click **Connect** on
the Drive pop-up. Wait for **✅ Setup complete**, then run the rest top to bottom.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os, sys, glob

print("1/3  installing mne ...")
get_ipython().system('pip install -q "mne==1.10.1"')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  connecting Google Drive for the data ...")
from google.colab import drive
drive.mount("/content/drive")
hits = sorted(glob.glob("/content/drive/MyDrive/**/synapse_preprocessed.pkl", recursive=True))
assert hits, (
    "Could not find synapse_preprocessed.pkl in your Drive.\n"
    "Open the shared camp folder, click 'Add shortcut to Drive', put the shortcut "
    "in 'My Drive', then run this cell again."
)
os.environ["CAMP_DATA_PATH"] = hits[0]
os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
print(f"\n\u2705 Setup complete. Using data at: {hits[0]}")
print("Your figures will be saved to Drive > DecodingBrain_outputs.")


# Week 3 · Tier 2 — Temporal Dynamics (When?)

**Research Goal 5 (the "when" part):**

> Do group differences appear *immediately* when the sound starts, or do they
> *build up* over time?

So far we used the whole sound window (0–2 s) as one lump. Now we slice each
trial into time windows — **early, mid, late, and after** the sound — and watch
the effect size evolve. We'll also use the HLT task to see how the brain scales
with **loudness**.

### By the end of this notebook you will be able to
1. Compute a feature in several time windows
2. Build an effect-size-over-time profile
3. Make a "loudness response curve" from the HLT task
4. Reason about the timing and intensity of neural responses

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import camp_utils as cu

data = cu.load_camp_data(verbose=False)

## 1. The time windows
`cu.get_time_windows(task)` gives named slices of each trial:

In [ ]:
windows = cu.get_time_windows("let")
for name, (start, end) in windows.items():
    print(f"  {name:11s}: {start:.1f} to {end:.1f} s")

## 2. Effect size in each window
For a feature, we compute Hedges' g (EXP vs CTRL) **separately in each window**.
We compare the same windows in order: early → mid → late → after the sound.

In [ ]:
def group_values(data, group, task, band, period):
    window = cu.get_time_windows(task)[period]
    vals = [cu.band_power_db(ep, band, window)
            for _, ep in cu.iter_subjects(data, group, task)]
    return [v for v in vals if not np.isnan(v)]

ordered_periods = ["early_stim", "mid_stim", "late_stim", "poststim"]
band, task = "gamma", "let"

effect_over_time = []
for period in ordered_periods:
    exp = group_values(data, "exp", task, band, period)
    ctrl = group_values(data, "ctrl", task, band, period)
    g = cu.hedges_g(exp, ctrl)
    effect_over_time.append(g)
    print(f"  {period:11s}: Hedges' g = {g:+.2f}")

## 3. Plot the temporal profile
Does the effect grow, shrink, or flip sign over the course of the trial?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(range(len(ordered_periods)), effect_over_time, "o-",
        color=cu.EXP_COLOR, linewidth=2, markersize=9)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
for thr in (-0.8, 0.8):
    ax.axhline(thr, color="lightgray", linestyle=":", linewidth=0.8)
ax.set_xticks(range(len(ordered_periods)))
ax.set_xticklabels(["early\n(0–0.5s)", "mid", "late", "after\nsound"])
ax.set_ylabel("Effect size (Hedges' g)")
ax.set_title(f"When does the group difference appear?  ({task.upper()} {band})")
plt.tight_layout()
plt.savefig(cu.save_path("tier2_temporal.png"), dpi=300, bbox_inches="tight")
plt.show()

### ✏️ Your turn #1 — make it a heatmap over all bands
A nicer figure: rows = bands, columns = time windows, color = effect size. Fill
in the calculation inside the loop.

In [ ]:
grid = pd.DataFrame(index=cu.BAND_ORDER, columns=ordered_periods, dtype=float)
for b in cu.BAND_ORDER:
    for period in ordered_periods:
        exp = group_values(data, "exp", task, b, period)
        ctrl = group_values(data, "ctrl", task, b, period)
        # TODO: compute Hedges' g and store it
        grid.loc[b, period] = np.nan   # replace with cu.hedges_g(exp, ctrl)

cu.check(grid.notna().all().all(),
         "Effect-size-over-time grid is filled.",
         "Store cu.hedges_g(exp, ctrl) in grid.loc[b, period].")

vmax = np.nanmax(np.abs(grid.values))
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(grid.values.astype(float), cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(len(ordered_periods)))
ax.set_xticklabels(ordered_periods, rotation=20)
ax.set_yticks(range(len(cu.BAND_ORDER))); ax.set_yticklabels(cu.BAND_ORDER)
for i in range(len(cu.BAND_ORDER)):
    for j in range(len(ordered_periods)):
        ax.text(j, i, f"{grid.iloc[i, j]:+.1f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, label="Hedges' g (EXP vs CTRL)")
ax.set_title(f"Effect size over time × band  ({task.upper()})")
plt.tight_layout()
plt.show()

## 4. Loudness response curve (HLT)
The HLT task plays the same kind of sound at **5 loudness levels** (3, 5, 10, 20,
40 dB). The "central gain" theory predicts sound-sensitive brains over-respond as
loudness climbs. Let's compute the gamma response at each level, per group.

In [ ]:
from camp_utils import HLT_INTENSITIES

def hlt_level_power(data, group, level, band="gamma"):
    """Mean gamma dB change for HLT trials at one intensity level, across a group."""
    window = cu.get_time_windows("hlt")["full_stim"]
    vals = []
    for subject, ep in cu.iter_subjects(data, group, "hlt"):
        # pick only the trials at this loudness (event names contain e.g. '10dB')
        names = [k for k in ep.event_id if level in k]
        if not names:
            continue
        sub_ep = ep[names]
        db = cu.band_power_db(sub_ep, band, window)
        if not np.isnan(db):
            vals.append(db)
    return np.mean(vals) if vals else np.nan

exp_curve = [hlt_level_power(data, "exp", lv) for lv in HLT_INTENSITIES]
ctrl_curve = [hlt_level_power(data, "ctrl", lv) for lv in HLT_INTENSITIES]

x = [int(lv.replace("dB", "")) for lv in HLT_INTENSITIES]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(x, exp_curve, "o-", color=cu.EXP_COLOR, label="EXP", linewidth=2)
ax.plot(x, ctrl_curve, "o-", color=cu.CTRL_COLOR, label="CTRL", linewidth=2)
ax.set_xlabel("Sound level (dB)")
ax.set_ylabel("Gamma change (dB)")
ax.set_title("Loudness response curve (HLT)")
ax.legend()
plt.tight_layout()
plt.savefig(cu.save_path("tier2_loudness_curve.png"), dpi=300, bbox_inches="tight")
plt.show()

### ✏️ Your turn #2 — interpret
Look at your loudness curve. As the sound gets louder:
- Does the EXP curve rise **faster/steeper** than CTRL? (That would support
  central gain.)
- Or do they track together?

Write 2 sentences (as a comment) describing what you see. Remember n is small —
don't overclaim.
YOUR ANSWER:

## 🎯 Wrap-up (Tier 2)
You added the dimension of **time** and **intensity**. "The difference appears
early and grows" or "the EXP brain ramps up steeply with loudness" are exactly
the kind of mechanistic stories that make a strong poster.

➡️ If you're done with Tier 2, peek at Tier 3 (Notebooks 13–14) for the
machine-learning challenge.